In [2]:
pip install -U jupyterlab_widgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import GPT2LMHeadModel, GPT2Config

torch.manual_seed(1337)

In [3]:
model = GPT2LMHeadModel.from_pretrained("gpt2")

print(model)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [4]:
config = model.config

print(config)

GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.5.4",
  "use_cache": true,
  "vocab_size": 50257
}



In [5]:
print("Vocabulary size :", config.vocab_size)
print("Context length  :", config.n_positions)
print("Embedding dim   :", config.n_embd)
print("Layers          :", config.n_layer)
print("Attention heads :", config.n_head)

Vocabulary size : 50257
Context length  : 1024
Embedding dim   : 768
Layers          : 12
Attention heads : 12


In [6]:
num_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {num_params:,}")
print(f"Total parameters: {num_params / 1e6:.2f}M")

Total parameters: 124,439,808
Total parameters: 124.44M


In [7]:
for name, module in model.named_modules():
    if name:
        print(name, "->", module.__class__.__name__)

transformer -> GPT2Model
transformer.wte -> Embedding
transformer.wpe -> Embedding
transformer.drop -> Dropout
transformer.h -> ModuleList
transformer.h.0 -> GPT2Block
transformer.h.0.ln_1 -> LayerNorm
transformer.h.0.attn -> GPT2Attention
transformer.h.0.attn.c_attn -> Conv1D
transformer.h.0.attn.c_proj -> Conv1D
transformer.h.0.attn.attn_dropout -> Dropout
transformer.h.0.attn.resid_dropout -> Dropout
transformer.h.0.ln_2 -> LayerNorm
transformer.h.0.mlp -> GPT2MLP
transformer.h.0.mlp.c_fc -> Conv1D
transformer.h.0.mlp.c_proj -> Conv1D
transformer.h.0.mlp.act -> NewGELUActivation
transformer.h.0.mlp.dropout -> Dropout
transformer.h.1 -> GPT2Block
transformer.h.1.ln_1 -> LayerNorm
transformer.h.1.attn -> GPT2Attention
transformer.h.1.attn.c_attn -> Conv1D
transformer.h.1.attn.c_proj -> Conv1D
transformer.h.1.attn.attn_dropout -> Dropout
transformer.h.1.attn.resid_dropout -> Dropout
transformer.h.1.ln_2 -> LayerNorm
transformer.h.1.mlp -> GPT2MLP
transformer.h.1.mlp.c_fc -> Conv1D
tran

In [8]:
block = model.transformer.h[0]

print(block)

GPT2Block(
  (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attn): GPT2Attention(
    (c_attn): Conv1D(nf=2304, nx=768)
    (c_proj): Conv1D(nf=768, nx=768)
    (attn_dropout): Dropout(p=0.1, inplace=False)
    (resid_dropout): Dropout(p=0.1, inplace=False)
  )
  (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (mlp): GPT2MLP(
    (c_fc): Conv1D(nf=3072, nx=768)
    (c_proj): Conv1D(nf=768, nx=3072)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)


In [9]:
print("Token embedding:", model.transformer.wte.weight.shape)
print("Position embedding:", model.transformer.wpe.weight.shape)

print("Attention QKV:", block.attn.c_attn.weight.shape)
print("Attention output:", block.attn.c_proj.weight.shape)

print("MLP first layer:", block.mlp.c_fc.weight.shape)
print("MLP second layer:", block.mlp.c_proj.weight.shape)

Token embedding: torch.Size([50257, 768])
Position embedding: torch.Size([1024, 768])
Attention QKV: torch.Size([768, 2304])
Attention output: torch.Size([768, 768])
MLP first layer: torch.Size([768, 3072])
MLP second layer: torch.Size([3072, 768])


In [11]:
torch.manual_seed(1337)

reference_model = GPT2LMHeadModel.from_pretrained(
    "openai-community/gpt2"
)

config = reference_model.config

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [12]:
print(reference_model.transformer.wte)
print(reference_model.transformer.wpe)

print("token embeddings :", reference_model.transformer.wte.weight.shape)
print("position embeddings:", reference_model.transformer.wpe.weight.shape)

Embedding(50257, 768)
Embedding(1024, 768)
token embeddings : torch.Size([50257, 768])
position embeddings: torch.Size([1024, 768])


In [14]:
class GPT2Embeddings(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd):
        super().__init__()

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)

    def forward(self, idx):
        B, T = idx.shape

        positions = torch.arange(
            T,
            device=idx.device
        )

        tok_emb = self.wte(idx)
        pos_emb = self.wpe(positions)

        return tok_emb + pos_emb

In [15]:
embeddings = GPT2Embeddings(
    config.vocab_size,
    config.n_positions,
    config.n_embd,
)

idx = torch.randint(
    0,
    config.vocab_size,
    (2, 8),
)

x = embeddings(idx)

print("input :", idx.shape)
print("output:", x.shape)

input : torch.Size([2, 8])
output: torch.Size([2, 8, 768])


In [16]:
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias=True):
        super().__init__()

        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(
            x,
            self.weight.shape,
            self.weight,
            self.bias,
            1e-5,
        )

In [17]:
ln = LayerNorm(config.n_embd)

x = torch.randn(2, 8, config.n_embd)
y = ln(x)

print("input :", x.shape)
print("output:", y.shape)
print("mean  :", y.mean().item())
print("std   :", y.std().item())

input : torch.Size([2, 8, 768])
output: torch.Size([2, 8, 768])
mean  : 0.0
std   : 1.0000356435775757


In [18]:
import math

In [19]:

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head

        # GPT-2 combines Q, K and V projections.
        self.c_attn = nn.Linear(
            n_embd,
            3 * n_embd
        )

        self.c_proj = nn.Linear(
            n_embd,
            n_embd
        )

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # Causal mask.
        mask = torch.tril(
            torch.ones(block_size, block_size)
        )

        self.register_buffer(
            "bias",
            mask.view(1, 1, block_size, block_size)
        )

    def forward(self, x):
        B, T, C = x.shape

        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # [B, T, C] -> [B, n_head, T, head_dim]
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Attention scores.
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # preventing attending to future tokens.
        att = att.masked_fill(
            self.bias[:, :, :T, :T] == 0,
            float("-inf")
        )

        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        # weighted combination of values.
        y = att @ v

        # [B, n_head, T, head_dim]
        # -> [B, T, n_head, head_dim]
        y = y.transpose(1, 2).contiguous()

        # merge heads.
        y = y.view(B, T, C)

        # output projection.
        y = self.c_proj(y)
        y = self.resid_dropout(y)

        return y
    

In [20]:
attention = CausalSelfAttention(
    n_embd=config.n_embd,
    n_head=config.n_head,
    block_size=config.n_positions,
)

x = torch.randn(2, 8, config.n_embd)

y = attention(x)
with torch.no_grad():
     qkv = attention.c_attn(x)
     print("QKV:", qkv.shape)
print("input :", x.shape)
print("output:", y.shape)

QKV: torch.Size([2, 8, 2304])
input : torch.Size([2, 8, 768])
output: torch.Size([2, 8, 768])


## GPT 2 MLP

In [21]:
class GPT2MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()

        self.c_fc = nn.Linear(
            n_embd,
            4 * n_embd
        )

        self.c_proj = nn.Linear(
            4 * n_embd,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = F.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)

        return x

In [22]:
mlp = GPT2MLP(config.n_embd)

x = torch.randn(2, 8, config.n_embd)
y = mlp(x)

print(y.shape)

torch.Size([2, 8, 768])


In [23]:
class GPT2Block(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        block_size,
        dropout=0.0,
    ):
        super().__init__()

        self.ln_1 = LayerNorm(n_embd)

        self.attn = CausalSelfAttention(
            n_embd,
            n_head,
            block_size,
            dropout,
        )

        self.ln_2 = LayerNorm(n_embd)

        self.mlp = GPT2MLP(
            n_embd,
            dropout,
        )

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))

        return x

In [24]:
block = GPT2Block(
    config.n_embd,
    config.n_head,
    config.n_positions,
)

x = torch.randn(2, 8, config.n_embd)
y = block(x)

print(y.shape)

torch.Size([2, 8, 768])


In [25]:
for name, param in block.named_parameters():
    print(f"{name:30s} {tuple(param.shape)}")

ln_1.weight                    (768,)
ln_1.bias                      (768,)
attn.c_attn.weight             (2304, 768)
attn.c_attn.bias               (2304,)
attn.c_proj.weight             (768, 768)
attn.c_proj.bias               (768,)
ln_2.weight                    (768,)
ln_2.bias                      (768,)
mlp.c_fc.weight                (3072, 768)
mlp.c_fc.bias                  (3072,)
mlp.c_proj.weight              (768, 3072)
mlp.c_proj.bias                (768,)


## Weight-tied language model head

In [27]:
class GPT2(nn.Module):
    def __init__(
        self,
        vocab_size,
        block_size,
        n_layer,
        n_head,
        n_embd,
        dropout=0.0,
    ):
        super().__init__()

        self.block_size = block_size

        self.transformer = nn.ModuleDict({
            "wte": nn.Embedding(vocab_size, n_embd),
            "wpe": nn.Embedding(block_size, n_embd),
            "h": nn.ModuleList([
                GPT2Block(
                    n_embd=n_embd,
                    n_head=n_head,
                    block_size=block_size,
                    dropout=dropout,
                )
                for _ in range(n_layer)
            ]),
            "ln_f": LayerNorm(n_embd),
        })

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False,
        )

        # tying token embeddings and output projection weights.
        self.lm_head.weight = self.transformer["wte"].weight

    def forward(self, idx):
        B, T = idx.shape

        assert T <= self.block_size, (
            f"sequence length {T} exceeds block size "
            f"{self.block_size}"
        )

        tok_emb = self.transformer["wte"](idx)

        pos = torch.arange(
            T,
            device=idx.device,
        )

        pos_emb = self.transformer["wpe"](pos)

        x = tok_emb + pos_emb

        for block in self.transformer["h"]:
            x = block(x)

        x = self.transformer["ln_f"](x)

        logits = self.lm_head(x)

        return logits

In [28]:
model = GPT2(
    vocab_size=config.vocab_size,
    block_size=config.n_positions,
    n_layer=config.n_layer,
    n_head=config.n_head,
    n_embd=config.n_embd,
)

idx = torch.randint(
    0,
    config.vocab_size,
    (2, 16),
)

logits = model(idx)

print("input :", idx.shape)
print("logits:", logits.shape)

input : torch.Size([2, 16])
logits: torch.Size([2, 16, 50257])


In [29]:
num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"parameters: {num_params:,}")
print(f"parameters: {num_params / 1e6:.2f}M")

parameters: 124,439,808
parameters: 124.44M


In [30]:
print(
    model.transformer["wte"].weight
    is model.lm_head.weight
)

True


In [31]:
print(model)

GPT2(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): GPT2MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [32]:
print(reference_model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [34]:
from transformers import  GPT2TokenizerFast
tokenizer = GPT2TokenizerFast.from_pretrained(
    "openai-community/gpt2"
)
print("vocab size:", tokenizer.vocab_size)



tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocab size: 50257


In [35]:
print("model vocab size:", config.vocab_size)
print("tokenizer vocab size:", tokenizer.vocab_size)


text = "Hello, my name is Abhijeet."

tokens = tokenizer.encode(text)

print(tokens)


decoded = tokenizer.decode(tokens)

print(decoded)


for token_id in tokens:
    token_bytes = tokenizer.decode([token_id])
    print(token_id, repr(token_bytes))
    
    
    
    
tokens = tokenizer.encode(text)

idx = torch.tensor(
    [tokens],
    dtype=torch.long,
)

print(idx.shape)
print(idx)

model vocab size: 50257
tokenizer vocab size: 50257
[15496, 11, 616, 1438, 318, 2275, 71, 2926, 68, 316, 13]
Hello, my name is Abhijeet.
15496 'Hello'
11 ','
616 ' my'
1438 ' name'
318 ' is'
2275 ' Ab'
71 'h'
2926 'ij'
68 'e'
316 'et'
13 '.'
torch.Size([1, 11])
tensor([[15496,    11,   616,  1438,   318,  2275,    71,  2926,    68,   316,
            13]])


In [37]:
with torch.no_grad():
    logits = model(idx)

print(logits.shape)


torch.Size([1, 11, 50257])
